In [4]:
%pip install hail

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 1. Import SparkSession
from typing import List
from pandas import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 2. Initialize SparkSession
spark: SparkSession = SparkSession.builder.appName("ORC Example").master("local[*]").getOrCreate()

# 3. Create sample data
data: list[tuple[str, int]] = [("a", 1), ("b", 2)]

schema = StructType([
    StructField("dim1", StringType(), True),
    StructField("dim2", IntegerType(), True)
])

df: DataFrame = spark.createDataFrame(data, schema)

# 4. Write data to ORC file
output_path = "/tmp/example.orc"
df.write.mode("overwrite").orc(output_path)

# 5. Read ORC file and calculate sum of dim2 column
df_orc = spark.read.orc(output_path)

# 7. Stop Spark session (commented for notebook use)
# spark.stop()


In [2]:



# Calculate aggregation and create new dataframe
agg_df = df_orc.agg({"dim2": "sum"})

# Save aggregation result to new ORC file
digest_path = "/workspaces/tech-spikes/pyspark-orc-examples/result"
agg_df.write.mode("overwrite").orc(digest_path)

In [3]:
# Read and display the original data
print("Original data:")
original_df = spark.read.orc(output_path)
original_df.show()

print("\nAggregation result:")
result_df = spark.read.orc(digest_path)
result_df.show()

Original data:
+----+----+
|dim1|dim2|
+----+----+
|   b|   2|
|   a|   1|
+----+----+


Aggregation result:
+---------+
|sum(dim2)|
+---------+
|        3|
+---------+

+----+----+
|dim1|dim2|
+----+----+
|   b|   2|
|   a|   1|
+----+----+


Aggregation result:
+---------+
|sum(dim2)|
+---------+
|        3|
+---------+

